In [21]:
"""from google.colab import drive
drive.mount('/content/drive')"""

"from google.colab import drive\ndrive.mount('/content/drive')"

In [22]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [23]:
df = pd.read_csv("../sample data/fashion-mnist_train.csv")

In [24]:
torch.manual_seed(42)

In [25]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device is {device}")

Device is cuda


In [26]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
X = df.drop("label", axis=1).to_numpy()
type(X)
#X.head()
y = df["label"].values

In [28]:
X, y

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(60000, 784)),
 array([2, 9, 6, ..., 8, 8, 7], shape=(60000,)))

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=.2, random_state=42)

X_train = X_train/255
X_test = X_test/255

In [30]:
class customdata(Dataset):
    def __init__(self, features, labels):

        self.features = torch.tensor(features, dtype = torch.float32).reshape(-1, 1,28, 28)    # unknown Batch size, Channels, Hight, Weidth
        self.labels = torch.tensor(labels, dtype = torch.long)
    
    def __len__(self):
        return self.features.shape[0]
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [31]:
train_dataset = customdata(X_train, y_train)

test_dataset = customdata(X_test, y_test)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory= True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True, pin_memory= True)

In [32]:
len(train_dataloader)

1500

In [33]:
class MyNN(nn.Module):
    def __init__(self, input_features):

        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=input_features, out_channels=32, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.BatchNorm2d(32), # SHOULD Match the it's Conv2d's OUTPUT Channels/ Filters number,
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding="same"),  # SHOULD Match the 1st Conv2d's OUTPUT Channels/ Filter's number
            nn.ReLU(),
            nn.BatchNorm2d(64), # Should match its Conv2d Layer's Output Channel/ Filters
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),

            #nn.LazyLinear(128),
            nn.Linear(64*7*7,128) ,# 128 is the output size (number of neurons). The input size is inferred automatically during the first forward pass. Manually, we would write nn.Linear(64 * 7 * 7, 128).
            #Output shape calculation after self.features:
            #Input Image           : (1, 28, 28)
            #Conv2d(1 -> 32)       : (32, 28, 28)   # 'same' padding keeps H and W unchanged
            #MaxPool2d(2, 2)       : (32, 14, 14)
            # kernel_size=2 and stride=2 halve the height and width (28 -> 14)
            #Conv2d(32 -> 64)      : (64, 14, 14)
            #MaxPool2d(2, 2)       : (64, 7, 7)
            # kernel_size=2 and stride=2 halve the height and width again (14 -> 7)
            #Flatten() converts (64, 7, 7) into:
            #64 * 7 * 7 = 3136 input features."""
            nn.ReLU(),
            nn.Dropout(p = .4),

            nn.Linear(128, 64),    # here input is 128 as it should match Previous's Linear's Output
            nn.ReLU(),
            nn.Dropout(p=.4),


            nn.Linear(64, 10))
            

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [34]:
learning_rate = 0.01
epochs = 100

In [35]:
model = MyNN(1)  # 1 as we have Black and White image & 3 for RGB
model.to(device)

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [36]:
critation = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [37]:
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features , batch_labels in train_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        output = model(batch_features)

        loss = critation(output, batch_labels)


        #backpass
        optimizer.zero_grad()
        loss.backward()


        #upate
        optimizer.step()

        total_epoch_loss = total_epoch_loss+loss.item()
    avg_loss = total_epoch_loss/len(train_dataloader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.6449703852335612
Epoch: 2 , Loss: 0.3843186562558015
Epoch: 3 , Loss: 0.32638632536431156
Epoch: 4 , Loss: 0.29047480096419653
Epoch: 5 , Loss: 0.26515363139907516
Epoch: 6 , Loss: 0.24613139249011876
Epoch: 7 , Loss: 0.23077783796625834
Epoch: 8 , Loss: 0.21190809412052233
Epoch: 9 , Loss: 0.20257316580725213
Epoch: 10 , Loss: 0.19359854978943863
Epoch: 11 , Loss: 0.17841900200396776
Epoch: 12 , Loss: 0.17097683236189187
Epoch: 13 , Loss: 0.16016317635122687
Epoch: 14 , Loss: 0.14904705005170157
Epoch: 15 , Loss: 0.14549349761381744
Epoch: 16 , Loss: 0.13480047553374122
Epoch: 17 , Loss: 0.12639724473531047
Epoch: 18 , Loss: 0.12259090473332132
Epoch: 19 , Loss: 0.113226802417698
Epoch: 20 , Loss: 0.10810563427240898
Epoch: 21 , Loss: 0.10528773803100921
Epoch: 22 , Loss: 0.09849583480320871
Epoch: 23 , Loss: 0.09609263050571705
Epoch: 24 , Loss: 0.08553151038723687
Epoch: 25 , Loss: 0.08341392594455586
Epoch: 26 , Loss: 0.08277753678072865
Epoch: 27 , Loss: 0.07980

In [38]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [39]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        output = model(batch_features)

        val, predicted_index = torch.max(output, 1)
        total = total + batch_features.shape[0]
        correct = correct + (predicted_index == batch_labels).sum().item()
print(correct/total)

0.9249166666666667


In [40]:
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in train_dataloader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        output = model(batch_features)

        val, predicted_index = torch.max(output, 1)
        total = total + batch_features.shape[0]
        correct = correct + (predicted_index == batch_labels).sum().item()
print(correct/total)

0.9996666666666667
